# Bootstrap vs Parametric Simulation of Financial Returns

This notebook compares two approaches for simulating financial return series:

1. **Bootstrap** – non-parametric resampling of historical returns.
2. **Parametric** – drawing from a fitted Normal or Student-t distribution.

We load a synthetic dataset, fit the models, run Monte Carlo simulations, and compare the results visually and numerically.

In [ ]:
import sys
import os

# Make sure the src package is importable when running from the notebooks/ directory
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
# If already at project root, the above is a no-op; add project root explicitly
project_root = os.path.abspath(os.path.join(os.path.dirname(''), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load Example Data

In [ ]:
# Resolve the data path relative to the notebook location
notebook_dir = os.getcwd()
data_path = os.path.join(notebook_dir, '..', 'data', 'example_returns.csv')
data_path = os.path.normpath(data_path)

df = pd.read_csv(data_path, parse_dates=['date'])
df.set_index('date', inplace=True)

print(f'Loaded {len(df)} observations from {df.index.min().date()} to {df.index.max().date()}')
df.head()

In [ ]:
returns = df['return'].values
print(f'Mean: {returns.mean():.6f}  |  Std: {returns.std():.6f}  |  Min: {returns.min():.6f}  |  Max: {returns.max():.6f}')

### Historical return series & volatility clustering

In [ ]:
from src.utils.plotting import plot_volatility_clustering

fig = plot_volatility_clustering(returns, window=21, title='Historical Returns — Volatility Clustering')
plt.show()

## 2. Bootstrap Simulation

In [ ]:
from src.models.bootstrap import simulate_bootstrap
from src.simulation.monte_carlo import run_monte_carlo

N_STEPS = 252   # one trading year
N_PATHS = 500
SEED = 42

boot_result = run_monte_carlo(
    simulate_bootstrap,
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    returns=returns,
    random_state=SEED,
)

print('Bootstrap returns shape :', boot_result['returns'].shape)
print('Cumulative returns shape:', boot_result['cum_returns'].shape)

## 3. Parametric Simulation

In [ ]:
from src.models.parametric import fit_normal, fit_student_t, simulate_parametric

params_norm = fit_normal(returns)
params_t    = fit_student_t(returns)

print('Normal fit     :', params_norm)
print('Student-t fit  :', params_t)

In [ ]:
norm_result = run_monte_carlo(
    simulate_parametric,
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    params=params_norm,
    random_state=SEED,
)

t_result = run_monte_carlo(
    simulate_parametric,
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    params=params_t,
    random_state=SEED,
)

print('Normal sim  shape:', norm_result['returns'].shape)
print('Student-t sim shape:', t_result['returns'].shape)

## 4. Comparison

### 4.1 Sample Paths

In [ ]:
from src.utils.plotting import plot_sample_paths

fig = plot_sample_paths(
    {
        'Bootstrap':    boot_result['cum_returns'],
        'Normal':       norm_result['cum_returns'],
        'Student-t':    t_result['cum_returns'],
    },
    n_display=50,
    title='Simulated Cumulative Return Paths',
    figsize=(15, 5),
)
plt.show()

### 4.2 Return Distributions

In [ ]:
from src.utils.plotting import plot_return_histogram

fig = plot_return_histogram(
    {
        'Historical':  returns,
        'Bootstrap':   boot_result['returns'].ravel(),
        'Normal':      norm_result['returns'].ravel(),
        'Student-t':   t_result['returns'].ravel(),
    },
    bins=80,
    title='Distribution of Simulated Returns vs Historical',
)
plt.show()

### 4.3 Moment Comparison

In [ ]:
from src.utils.statistics import compare_statistics

stats_df = compare_statistics({
    'Historical':  returns,
    'Bootstrap':   boot_result['returns'].ravel(),
    'Normal':      norm_result['returns'].ravel(),
    'Student-t':   t_result['returns'].ravel(),
})

stats_df.round(6)

## 5. Summary

| Method | Strengths | Weaknesses |
|---|---|---|
| **Bootstrap** | Preserves empirical distribution exactly; no distributional assumption | Cannot extrapolate beyond observed range; ignores time dependence |
| **Normal** | Simple; analytically tractable | Under-estimates tail risk; symmetric |
| **Student-t** | Heavier tails than Normal; better for financial returns | Still symmetric; single static distribution |

Future extensions: GARCH, HAR, GBM, Heston, VAE/GAN generative models.